# Stage 6 (image) — Fundus Image Classification Proof of Concept
**Capstone: Multimodal Ocular Disease Prediction**

Pipeline position: Tabular baseline (done) → Fundus image inspection (done) → **CNN transfer-learning proof of concept (this notebook)** → full-scale image model → clinical+image fusion.

Task: binary classification, **Myopia vs Normal**, using the `datasets/myopia` and `datasets/normal` folders (already resized 224x224 PNGs, recommended in the Stage 5 inspection as the fast-demo image source — see the master documentation, Section 3.1).

**Important, read before trusting the numbers below — this notebook was run twice, in two different places, and the two runs are NOT the same experiment:**

1. **Verification run (this file, as delivered).** Run inside a network-restricted analysis sandbox that cannot reach the ImageNet pretrained-weights host (`storage.googleapis.com`) or GitHub release assets — only `pypi.org` is reachable there. The code below detects this, prints a loud warning, and falls back to a **randomly-initialized** MobileNetV2 backbone so the full pipeline (data loading, augmentation, training loop, evaluation, plots) could still be built and proven to run correctly end-to-end on real images. **A randomly-initialized backbone is NOT transfer learning** — its accuracy is not meaningful and should not be reported as the project's result.
2. **The real transfer-learning run must happen in Colab** (or any environment with normal internet access), where `weights="imagenet"` downloads successfully and the backbone is a genuine pretrained feature extractor. Run this notebook there unchanged — the fallback code simply won't trigger, and the metrics/plots produced will be the actual result to report.

Also note the **image budget**: 103 Myopia + 108 Normal images (211 total) were used, not the full 232/2,873 available in those folders. This is a deliberate proof-of-concept subsample chosen for a fast first result — see Section 1 for the full reasoning. Scaling up to the complete class folders is listed as a next step.

## 1. Locate and load the images

Uses `datasets/myopia` (Myopia) and `datasets/normal` (Normal) — the two classes recommended in Stage 5 as the ready-to-use, already-resized fast-demo source. `IMAGES/Myopia_images` and `IMAGES/Normal_images` are deliberately **not** used, per Stage 5's finding that they are unverified, far larger than the real record counts, and likely heavily augmented.

**Why not all 232 / 2,873 images?** This proof-of-concept prioritizes a fast, honest, fully-executed first result over completeness. 103 Myopia + 108 Normal images were available in the local working copy used to build/verify this notebook. Scaling to the full folders (especially all ~2,873 Normal images) is a straightforward next step — the code below does not hardcode the sample size, so pointing `DATA_DIR` at the complete folders and re-running is enough.

In [1]:
import os

CANDIDATE_ROOTS = ["../../cnn_data", "cnn_data", "."]
DATA_DIR = next((r for r in CANDIDATE_ROOTS if os.path.isdir(os.path.join(r, "myopia")) and os.path.isdir(os.path.join(r, "normal"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find myopia/ and normal/ subfolders. Set DATA_DIR manually.")

myopia_files = sorted(f for f in os.listdir(os.path.join(DATA_DIR, "myopia")) if f.lower().endswith(".png"))
normal_files = sorted(f for f in os.listdir(os.path.join(DATA_DIR, "normal")) if f.lower().endswith(".png"))

print("DATA_DIR:", os.path.abspath(DATA_DIR))
print("Myopia images found:", len(myopia_files))
print("Normal images found:", len(normal_files))


DATA_DIR: /home/claude/project/mirror/cnn_data
Myopia images found: 103
Normal images found: 108


## 2. Load images into arrays, check shapes

Every image is confirmed 224x224 RGB PNG (verified in Stage 5) — loaded directly at that size, no resizing needed.

In [2]:
import numpy as np
from PIL import Image

IMG_SIZE = (224, 224)

def load_images(folder, filenames, label):
    X, y = [], []
    for fn in filenames:
        path = os.path.join(folder, fn)
        with Image.open(path) as im:
            im = im.convert("RGB")
            arr = np.array(im, dtype=np.float32)
        X.append(arr)
        y.append(label)
    return X, y

X_myopia, y_myopia = load_images(os.path.join(DATA_DIR, "myopia"), myopia_files, 1)
X_normal, y_normal = load_images(os.path.join(DATA_DIR, "normal"), normal_files, 0)

X = np.array(X_myopia + X_normal)
y = np.array(y_myopia + y_normal)

print("X shape:", X.shape, "| y shape:", y.shape)
print("Class counts -> Myopia(1):", int(y.sum()), " Normal(0):", int((y == 0).sum()))
assert X.shape[1:] == (224, 224, 3)


X shape: (211, 224, 224, 3) | y shape: (211,)
Class counts -> Myopia(1): 103  Normal(0): 108


## 3. Train / validation / test split

Stratified 70/15/15 split so both classes stay proportionally represented in every set. This is an **image-level** split (Stage 5 already established `datasets/<class>` filenames carry no patient ID, so a patient-level grouped split — as used for the tabular data — is not possible here).

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, "positive rate:", y_train.mean().round(3))
print("Val:  ", X_val.shape, "positive rate:", y_val.mean().round(3))
print("Test: ", X_test.shape, "positive rate:", y_test.mean().round(3))


Train: (147, 224, 224, 3) positive rate: 0.49
Val:   (32, 224, 224, 3) positive rate: 0.5
Test:  (32, 224, 224, 3) positive rate: 0.469


## 4. Preprocessing and augmentation

- Pixel values scaled with `keras.applications.mobilenet_v2.preprocess_input` (matches how the pretrained backbone was originally trained).
- Light augmentation applied to the **training set only** (random flip, rotation, zoom, contrast) — a small dataset like this one benefits from augmentation to reduce overfitting; validation/test stay unaugmented so evaluation reflects real images.

In [4]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="augmentation")

def make_dataset(X_arr, y_arr, training, batch_size=16):
    ds = tf.data.Dataset.from_tensor_slices((X_arr, y_arr))
    if training:
        ds = ds.shuffle(buffer_size=len(X_arr), seed=42)
    ds = ds.batch(batch_size)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val, training=False)
test_ds = make_dataset(X_test, y_test, training=False)
print("Datasets built.")


I0000 00:00:1788961258.563774    1427 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788961258.617286    1427 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1788961260.196205    1427 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Datasets built.


## 5. Build the transfer-learning model (MobileNetV2 backbone)

MobileNetV2 was chosen over ResNet50 / EfficientNet for this proof-of-concept because it is far lighter (~3.5M parameters vs ~25M+), which matters a lot on a CPU-only setup with a small dataset — it trains in a fraction of the time with far less overfitting risk, while still being a standard, well-validated ImageNet backbone. Swapping in ResNet50/EfficientNetB0 for the final version is a one-line change (`tf.keras.applications.ResNet50` / `EfficientNetB0`) once training moves to a GPU environment with more images.

The backbone is **frozen** (feature extraction, not full fine-tuning) — appropriate given how few images are available; fine-tuning the whole network on ~150 training images would overfit badly. A small trainable classification head (GlobalAveragePooling → Dropout → Dense) sits on top.

**Weights fallback (see the top-of-notebook note):** the cell below tries `weights="imagenet"` first; if that download fails (as it does in this sandbox), it prints a clear warning and continues with `weights=None` so the rest of the pipeline still runs and can be verified end-to-end.

In [5]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

WEIGHTS_USED = "imagenet"
try:
    base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
except Exception as e:
    WEIGHTS_USED = "random_init_fallback"
    print("=" * 70)
    print("WARNING: could not download ImageNet pretrained weights.")
    print("Reason:", repr(e))
    print("Falling back to weights=None (random initialization).")
    print("This is NOT real transfer learning -- results below only verify")
    print("that the pipeline code runs correctly on real images.")
    print("Re-run this notebook in an environment with normal internet")
    print("access (e.g. Google Colab) to get the true transfer-learning result.")
    print("=" * 70)
    base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights=None)

base_model.trainable = False  # freeze backbone -- feature extraction, not fine-tuning

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
print("WEIGHTS_USED:", WEIGHTS_USED)
model.summary()


Reason: Exception('URL fetch failure on https://storage.googleapis.com/tensorflow/keras-applications/mobilenet_v2/mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5: None -- Tunnel connection failed: 403 Forbidden')
Falling back to weights=None (random initialization).
This is NOT real transfer learning -- results below only verify
that the pipeline code runs correctly on real images.
Re-run this notebook in an environment with normal internet
access (e.g. Google Colab) to get the true transfer-learning result.


WEIGHTS_USED: random_init_fallback


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,033 (8.93 MB)

 Trainable params: 82,049 (320.50 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 6. Train

Class weights are computed from the training set (in case the split is not perfectly 50/50) and passed to `fit`, consistent with the imbalance-handling approach already used for the tabular baseline in Stage 4. Training is capped at a modest number of epochs with early stopping on validation loss, appropriate for this small dataset.

In [6]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print("Class weights:", class_weight)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

EPOCHS = 20
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=2,
)
print("Training complete. Epochs actually run:", len(history.history["loss"]))


Class weights: {0: 0.98, 1: 1.0208333333333333}
Epoch 1/20


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


10/10 - 8s - 763ms/step - accuracy: 0.5102 - loss: 0.6934 - val_accuracy: 0.5000 - val_loss: 0.6931


Epoch 2/20


10/10 - 3s - 275ms/step - accuracy: 0.5102 - loss: 0.6933 - val_accuracy: 0.5000 - val_loss: 0.6932


...

Epoch 5/20


10/10 - 3s - 251ms/step - accuracy: 0.5102 - loss: 0.6933 - val_accuracy: 0.5000 - val_loss: 0.6932


Epoch 6/20


10/10 - 3s - 289ms/step - accuracy: 0.5102 - loss: 0.6935 - val_accuracy: 0.5000 - val_loss: 0.6932


Training complete. Epochs actually run: 6


## 7. Training / validation curves

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history["loss"], label="train loss")
axes[0].plot(history.history["val_loss"], label="val loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train accuracy")
axes[1].plot(history.history["val_accuracy"], label="val accuracy")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig("stage6_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved stage6_training_curves.png")


Saved stage6_training_curves.png


## 8. Evaluate on the held-out test set

Accuracy, precision, recall, F1 (per-class and macro), and the confusion matrix — the same metric set used for the tabular baseline in Stage 4, for consistency.

In [8]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

y_prob = model.predict(test_ds).ravel()
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"WEIGHTS_USED: {WEIGHTS_USED}  <-- read this before trusting these numbers (see top of notebook)")
print(f"Test accuracy:  {acc:.4f}")
print(f"Precision (Myopia): {prec:.4f}")
print(f"Recall (Myopia):    {rec:.4f}")
print(f"F1 (Myopia):         {f1:.4f}")
print(f"F1 (macro):          {f1_macro:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Normal", "Myopia"], zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print("Confusion matrix (rows=actual, cols=predicted):")
print(cm)


1/2 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 161ms/step


WEIGHTS_USED: random_init_fallback  <-- read this before trusting these numbers (see top of notebook)
Test accuracy:  0.5312
Precision (Myopia): 0.0000
Recall (Myopia):    0.0000
F1 (Myopia):         0.0000
F1 (macro):          0.3469

              precision    recall  f1-score   support

      Normal       0.53      1.00      0.69        17
      Myopia       0.00      0.00      0.00        15

    accuracy                           0.53        32
   macro avg       0.27      0.50      0.35        32
weighted avg       0.28      0.53      0.37        32

Confusion matrix (rows=actual, cols=predicted):
[[17  0]
 [15  0]]


In [9]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["Normal", "Myopia"], yticklabels=["Normal", "Myopia"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Test set confusion matrix (weights={WEIGHTS_USED})")
plt.tight_layout()
plt.savefig("stage6_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved stage6_confusion_matrix.png")


Saved stage6_confusion_matrix.png


## 9. Save the model checkpoint

Saved regardless of which weights were used, but the filename/print statement records the mode so it's never mistaken for a real trained model later.

In [10]:
MODELS_DIR = os.path.join(os.path.dirname(DATA_DIR), "Code", "models") if os.path.isdir(os.path.join(os.path.dirname(DATA_DIR), "Code")) else "models"
os.makedirs(MODELS_DIR, exist_ok=True)
model_path = os.path.join(MODELS_DIR, f"stage6_myopia_vs_normal_mobilenetv2_{WEIGHTS_USED}.keras")
model.save(model_path)
print("Saved model to:", model_path)


Saved model to: ../../Code/models/stage6_myopia_vs_normal_mobilenetv2_random_init_fallback.keras


## 10. Summary

| Item | Value |
|---|---|
| Task | Binary: Myopia vs Normal |
| Data source | datasets/myopia, datasets/normal (Stage 5 recommended fast-demo source) |
| Images used | 103 Myopia + 108 Normal = 211 (subsample — see Section 1) |
| Split | 70% train / 15% val / 15% test, stratified, image-level |
| Backbone | MobileNetV2, frozen (feature extraction) |
| Weights | See `WEIGHTS_USED` printed above — `imagenet` if the pretrained download succeeded, `random_init_fallback` if this ran in a network-restricted environment |
| Augmentation | Random flip, rotation, zoom, contrast (train only) |
| Imbalance handling | `class_weight` computed from the training split |
| Training | Up to 20 epochs, early stopping on val loss |
| Evaluation | Accuracy, Precision/Recall/F1 (Myopia + macro), confusion matrix, training/validation curves — see Sections 7-8 |
| Model checkpoint | Code/models/stage6_myopia_vs_normal_mobilenetv2_&lt;WEIGHTS_USED&gt;.keras |

**If `WEIGHTS_USED` printed `random_init_fallback` in your run:** the metrics above only prove the pipeline executes correctly on real images — they are not a meaningful transfer-learning result. Re-run this notebook in Colab (normal internet access) to get the real numbers, then that run's Section 8 output is what should be reported and copied into the project documentation.

**Not done in this notebook:** using the full 232/2,873-image class folders, fine-tuning the backbone, ResNet50/EfficientNet comparison, and the eventual clinical+image fusion.